In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)


In [ ]:
from eodgdl import load_eod

df_viv, df_hab, df_trips, df_legs = load_eod(Path("../data/"))

In [ ]:
df_hab.ponderador.sum()

In [ ]:
(
    df_viv.ponderador.sum(),
    (df_viv.ponderador * df_hab.groupby("folio_vivienda").size()).sum(),
)

In [ ]:
df_trips.ponderador.groupby(["folio_vivienda", "folio_habitante"]).nunique().loc[
    lambda s: s > 1
].shape

In [ ]:
df_viv[["ponderador"]].merge(
    df_hab[["ponderador"]], left_index=True, right_index=True, suffixes=("_viv", "_hab")
).merge(
    df_trips[["ponderador"]].rename(columns={"ponderador": "ponderador_trip"}),
    left_index=True,
    right_index=True,
).loc[37]

In [ ]:
df_viv.ingreso_mensual_hogar.value_counts()

In [ ]:
# This columns contain survey design info, not collected info
od_h_cols_survey_design = [
    "Fecha",
    "Persona del hogar que fue el informante principal",
    "En su hogar, ¿Estarían dispuestos a participar en este mismo tipo de estudio en el futuro?",
]

# This are columns with overlapping information with the census
od_h_cols_census_like = {
    "¿Cuántos ... tienen en esta vivienda? | Vehículos (autos o camionetas)": "num_auto",
    "¿Cuántos ... tienen en esta vivienda? | Motocicletas o motonetas": "num_moto",
    "¿Cuántos ... tienen en esta vivienda? | Bicicletas": "num_bici",
    "¿En su vivienda tienen acceso a internet?": "internet",
    "Incluyéndolo, ¿cuántas personas viven permanentemente en su vivienda contando a los bebés y personas adultas mayores?": "tamaño_hogar",
}

# This columns are survey only
od_h_cols_survey_only = {
    "En su opinión, ¿Cuál es el principal problema de movilidad en sus traslados en la zona donde vive?": "prob_movi",
    "Y con respecto al TRANSPORTE PÚBLICO, ¿Cuál considera que es el principal problema de la zona donde vive?": "prob_tp",
    "Esta casa es?": "estado_propiedad",
    "¿Ingresos mensuales de todas las personas que habitan en la vivienda y aportan gastos al hogar?": "ingresos",
    "¿Dónde estacionan los vehículos?": "estacionamiento",
    "Ponderador": "ExpansionFactor",
}

# This columns ID the row
od_h_cols_id = {
    "Municipio": "NOM_MUN",
    "Centralidad": "CENT",
    "AGEB": "AGEB",
    "Folio Vivienda": "HouseholdId",
}

# We additionally drop columns with no useful information for modelling.
# This are also the columns with NA values
od_h_cols_to_drop = [
    "prob_movi",
    "prob_tp",
    "estacionamiento",
]

In [ ]:
# Import eod survey files
od_h = (
    pd.read_csv(
        "../data/IMEPLAN_Base_Viviendas_Master.csv",
        encoding="latin",
        low_memory=False,
    )
    .drop(columns=od_h_cols_survey_design)
    .rename(columns=od_h_cols_census_like)
    .rename(columns=od_h_cols_survey_only)
    .rename(columns=od_h_cols_id)
    .drop(columns=od_h_cols_to_drop)
    .set_index("HouseholdId")
    .sort_index()
)
od_p = pd.read_csv(
    "../data/IMEPLAN_Base_Habitantes_Master.csv",
    encoding="latin",
    low_memory=False,
)
od_t = pd.read_csv(
    "../data/IMEPLAN_Base_Viajes_Master.csv", encoding="latin", low_memory=False
)

In [ ]:
od_h.columns

In [ ]:
od_h.isna().sum()

In [ ]:
od_p.columns